## Extending the Flask API You Built
- This notebook is a **companion to `Week5.ipynb`**. It does not repeat the theory but it gives you small, guided modifications to the files you already worked with.
- In addition to take-home mini challenge that pulls everything together (routing, JWT auth, and Docker).

### Before you start
- Make sure you completed `Week5.ipynb` first. You'll be editing files you already opened there: `docker.py`, `server.py`, `client.py`, and `project/routes/user_routes.py`.
- Keep the **Week5 directory** open in your terminal/editor, same as before.
- Each exercise below tells you **which file to open**, **exactly what to add**, and **what output you should see** when it works. If your output doesn't match, that's the bug to fix. Don't move on until it does.

### How this notebook is structured?
| Exercise | File you edit | Concept practiced |
|---|---|---|
| 1 | `docker.py` | Adding a new route (DELETE) |
| 2 | `project/routes/user_routes.py` | Input validation on a Blueprint route |
| 3 | `server.py` + `client.py` | Reading identity out of a JWT |
| 4 | `docker.py` + `Dockerfile.dev` | Rebuilding a Docker image after a code change |
| **Mini Challenge** (BONUS) | new files you create | Combining routing + JWT + Docker |


## **Problem 1:  Add a `DELETE` route**

> **File to open:** [`docker.py`](docker.py)

> Right now `docker.py` only has `GET /` and `GET /users`. Your task is to add a route that removes a user by `id`.

**Steps:**
1. Open [```docker.py```](docker.py).
2. Add the following route anywhere below the existing `/users` route:

```python
@app.route("/users/<int:user_id>", methods=["DELETE"])
def delete_user(user_id):
    global users_list
    users_list = [u for u in users_list if u["id"] != user_id]
    return jsonify({"message": f"User {user_id} deleted"})
```

3. Since `docker.py` currently builds the users list *inside* the `/users` function (not as a variable the app can reuse), change the top of the file so the list is defined **once**, outside any function, and reused by both routes:

```python
users_list = [
    {"id": 1, "name": "Harry"},
    {"id": 2, "name": "Styles"}
]

@app.route("/users")
def users():
    return jsonify(users_list)
```

4. Save the file, then in your terminal (Week5 directory):
```bash
python docker.py
```

**Expected output** → visiting these two links in order:
- `http://127.0.0.1:5000/users` → `[{"id": 1, "name": "Harry"}, {"id": 2, "name": "Styles"}]`
- Send a `DELETE` request to `http://127.0.0.1:5000/users/1` (use the cell below, or `curl -X DELETE http://127.0.0.1:5000/users/1`) → `{"message": "User 1 deleted"}`
- Refresh `http://127.0.0.1:5000/users` → `[{"id": 2, "name": "Styles"}]` (Harry is gone)

Run this cell **after** `python docker.py` is running in your terminal:


In [ ]:
import requests

base = "http://127.0.0.1:5000"
print("Before delete:", requests.get(f"{base}/users").json())
print("Delete response:", requests.delete(f"{base}/users/1").json())
print("After delete:", requests.get(f"{base}/users").json())


## **Problem 2: Validate input on a Blueprint route**

**File to open:** `project/routes/user_routes.py`

> Right now `add_user()` accepts *anything* sent to it → even an empty body. Your task is to reject incomplete data.

**Steps:**
1. Open `project/routes/user_routes.py`.
2. Replace the `add_user()` function with this version:

```python
@user_bp.route("/", methods=["POST"])
def add_user():
    data = request.get_json()

    # Reject if the body is missing or doesn't have both required fields
    if not data or "name" not in data or "email" not in data:
        return jsonify({"error": "Both 'name' and 'email' are required"}), 400

    users.append(data)
    return jsonify({"message": "User added"}), 201
```

3. In your terminal, from `project/`:
```bash
python app.py
```

**Expected output:**
- `POST http://127.0.0.1:5000/users/` with body `{"name": "Sam"}` (missing email) → **400** status, `{"error": "Both 'name' and 'email' are required"}`
- `POST http://127.0.0.1:5000/users/` with body `{"name": "Sam", "email": "sam@example.com"}` → **201** status, `{"message": "User added"}`

Test both cases with the cell below (run it after `python app.py` is running):


In [ ]:
import requests

base = "http://127.0.0.1:5000/users/"

# Case 1: missing field -> should fail with 400
bad = requests.post(base, json={"name": "Sam"})
print("Missing email ->", bad.status_code, bad.json())

# Case 2: complete data -> should succeed with 201
good = requests.post(base, json={"name": "Sam", "email": "sam@example.com"})
print("Complete data ->", good.status_code, good.json())


## **Problem 3: Read identity out of the JWT**

> **Files to open:** [```server.py```](server.py) and [```client.py```](client.py)

`/protected` currently just returns a static message → it doesn't actually use *who* logged in. Your task is to add a `/profile` route that returns the username stored inside the token.

**Steps:**
1. Open `server.py`. Update the import line at the top to also bring in `get_jwt_identity`:

```python
from flask_jwt_extended import JWTManager, create_access_token, jwt_required, get_jwt_identity
```

2. Add this new route below `/protected`:

```python
@app.route("/profile", methods=["GET"])
@jwt_required()
def profile():
    current_user = get_jwt_identity()
    return jsonify({"logged_in_as": current_user})
```

3. Open `client.py` and add this block at the end of the file (after the existing `/protected` call):

```python
# --- Step 3: Call the new /profile route ---
profile_response = requests.get(f"{BASE_URL}/profile", headers=headers)

if profile_response.status_code == 200:
    print("Profile response:", profile_response.json())
else:
    print("Failed to access profile route:", profile_response.json())
```

4. In one terminal: `python server.py`. In a second terminal: `python client.py`.

**Expected output** (last line printed by `client.py`):
```
Profile response: {'logged_in_as': 'student'}
```

**Question:** where did the string `"student"` actually come from → trace it back to the line in `server.py` where the token was first created.


## **Probelm 4: Rebuild the Docker image after a code change**

> **Files to open:** `docker.py` (*with your Exercise 1 changes*) and [```Dockerfile.dev```](Dockerfile.dev)

> Docker images are **snapshots** → editing `docker.py` on your machine does *not* change a container you already built. You have to rebuild. 

**Steps:**
1. Add one more route to `docker.py` (the same file from Exercise 1):

```python
@app.route("/health")
def health():
    return jsonify({"status": "ok", "user_count": len(users_list)})
```

2. Rebuild the image → the same command as in Week5.ipynb, run again:
```bash
docker buildx build -t flask-api-dev -f Dockerfile.dev .
```

3. Run the container:
```bash
docker run -p 5000:5000 flask-api-dev
```

**Expected output:**
- `http://localhost:5000/health` → `{"status": "ok", "user_count": 2}`

**Question:** what would you have seen at `/health` if you had run `docker run` *without* rebuilding first? Why?


### Well done, Week 5 Exercises Complete 🎉!
- Don't forget to submit your [exercise notebook](Week5_Exercise.ipynb) on your private GitHub Repository (see [submission instructions](https://github.com/SRH-Heidelberg-University-ADSA/Advanced-Python-for-Data-Science/blob/main/SUBMISSION_INSTRUCTIONS.md)). The deadline is 19.07.2026 @ 23:59. 

- See you next time when we dive into `Cloud Deployment & DevOps`!
---

## Mini Challenge: Secured, Dockerized "Delete User" API (Bonus)

This challenge combines all three ideas from this notebook: **routing** (Ex. 1), **JWT authentication** (Ex. 3), and **Docker** (Ex. 4). Nothing here is copy-paste → you're assembling pieces you've already built separately.

### The goal
Build an API where:
- Anyone can `POST /login` with `{"username": "student", "password": "pass123"}` to get a JWT (reuse the logic from `server.py`).
- Anyone can `GET /users` **without** a token (public).
- Only someone with a valid JWT can `DELETE /users/<id>` (protected).
- The whole thing runs in a Docker container.

### Step-by-step instructions

**1. Create a new file `challenge_api.py`** in your Week5 directory. 
**Here is a skeleton → the `TODO`s are for you to fill in using what you did in the exercises above:**

```python
from flask import Flask, jsonify, request
from flask_jwt_extended import JWTManager, create_access_token, jwt_required

app = Flask(__name__)
app.config["JWT_SECRET_KEY"] = "super-secret-key"
jwt = JWTManager(app)

USER = {"username": "student", "password": "pass123"}

users_list = [
    {"id": 1, "name": "Harry"},
    {"id": 2, "name": "Styles"}
]

@app.route("/login", methods=["POST"])
def login():
    # TODO: copy the validation + token creation logic from server.py's login()
    pass

@app.route("/users", methods=["GET"])
def get_users():
    # TODO: return users_list as JSON → this route needs NO authentication
    pass

@app.route("/users/<int:user_id>", methods=["DELETE"])
@jwt_required()
def delete_user(user_id):
    # TODO: remove the matching user from users_list (see Exercise 1)
    # then return a confirmation message
    pass

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
```

**2. Test it locally first** (before Docker). Run `python challenge_api.py`, then write a small `challenge_client.py` (or reuse the cell below once the container is running) that:
   1. Logs in and gets a token.
   2. Tries `DELETE /users/1` **without** a token → should fail.
   3. Tries `DELETE /users/1` **with** the token → should succeed.

**3. Dockerize it.** Create `Dockerfile.challenge`, based on the pattern in `Dockerfile.dev`, but pointing at your new file:

```dockerfile
FROM python:3.10-slim
ENV PYTHONUNBUFFERED=1
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt flask-jwt-extended
COPY . .
EXPOSE 5000
CMD ["python", "challenge_api.py"]
```

**4. Build and run:**
```bash
docker buildx build -t flask-challenge -f Dockerfile.challenge .
docker run -p 5000:5000 flask-challenge
```

### Expected output (run the cell below once the container is up)
```
Public GET /users        -> 200 [{'id': 1, 'name': 'Harry'}, {'id': 2, 'name': 'Styles'}]
DELETE without token      -> 401 {'msg': 'Missing Authorization Header'}
Login                      -> 200 token received
DELETE with token          -> 200 {'message': 'User 1 deleted'}
GET /users after delete    -> 200 [{'id': 2, 'name': 'Styles'}]
```

If your numbers/messages match this shape (exact wording may differ slightly depending on how you wrote your messages), your challenge is complete. 🎉


In [ ]:
##################
# Step 3         #
##################

import requests

base = "http://127.0.0.1:5000"

# 1. Public route, no token needed
r = requests.get(f"{base}/users")
print("Public GET /users        ->", r.status_code, r.json())

# 2. Try deleting without a token -> should fail
r = requests.delete(f"{base}/users/1")
print("DELETE without token      ->", r.status_code, r.json())

# 3. Log in
login = requests.post(f"{base}/login", json={"username": "student", "password": "pass123"})
token = login.json().get("access_token")
print("Login                      ->", login.status_code, "token received" if token else "NO TOKEN")

# 4. Delete with token -> should succeed
headers = {"Authorization": f"Bearer {token}"}
r = requests.delete(f"{base}/users/1", headers=headers)
print("DELETE with token          ->", r.status_code, r.json())

# 5. Confirm the user is gone
r = requests.get(f"{base}/users")
print("GET /users after delete    ->", r.status_code, r.json())


### Questions
- Which route in your challenge API needed `@jwt_required()` and which didn't → why that split?
- What would happen if you swapped `EXPOSE 5000` for `EXPOSE 8000` in `Dockerfile.challenge` but left `app.run(..., port=5000)` unchanged in the code? Would the API still work from your host machine?
- Where is the JWT secret key currently stored in `challenge_api.py`, and why is that not safe for a real production deployment (hint: revisit the comment in `server.py`)?
